In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
%pip install semantic-link-labs

In [ ]:
# ---------------------------------------------------------------------------
# Disable IsAvailableInMdx on every eligible column in every semantic model
# in a workspace.
#
# Run each block below in its own Fabric notebook cell (the %pip install must
# be the first cell and will restart the Python session).
#
# Note: connect_semantic_model(readonly=False) calls model.SaveChanges() for
# you when the "with" block exits, so there is no explicit save here.
# ---------------------------------------------------------------------------

import re

import pandas as pd
import sempy.fabric as fabric
from sempy_labs.tom import connect_semantic_model

# None = the workspace the notebook is attached to. Otherwise set a name or id.
WORKSPACE = None

# Set to True for a dry run that reports what would change without saving.
WHAT_IF = False

# Models that are auto-generated or otherwise should not be touched.
SKIP_MODELS = set()

# The server rejects the entire save if a column that still needs an attribute
# hierarchy was disabled. When that happens we parse the offending column out
# of the error, add it to the keep-list, and retry.
MAX_RETRIES = 10

# After the <oii> tags are stripped the server writes the reference as
# 'Table[Column]' (the whole thing quoted). The optional quote also tolerates
# a 'Table'[Column] style reference.
_ERROR_COLUMN_RE = re.compile(r"The column '([^'\[]+)'?\[([^\]]+)\]")


def _columns_requiring_mdx(tom) -> dict:
    """Columns that must keep IsAvailableInMdx = True, mapped to the reason.

    Mirrors Microsoft's BPA rule SET_ISAVAILABLEINMDX_TO_TRUE_ON_NECESSARY_COLUMNS:
    a column needs an attribute hierarchy if it is used as a sort-by column,
    used in a hierarchy, used in a variation, or is itself sorted by another
    column.
    """

    required = {}

    def mark(column, reason):
        if column is not None:
            required.setdefault((column.Parent.Name, column.Name), reason)

    # Used as a hierarchy level.
    for lvl in tom.all_levels():
        mark(lvl.Column, "used in a hierarchy")

    for col in tom.all_columns():
        # The column is sorted by another column...
        if col.SortByColumn is not None:
            mark(col, "has a Sort By Column")
            # ...and that other column is the one doing the sorting.
            mark(col.SortByColumn, "used as a Sort By Column")

        # Used in a variation (e.g. a date table's default date hierarchy).
        try:
            for variation in col.Variations:
                mark(col, "used in a variation")
                mark(variation.DefaultColumn, "used in a variation")
                if variation.DefaultHierarchy is not None:
                    for lvl in variation.DefaultHierarchy.Levels:
                        mark(lvl.Column, "used in a variation")
        except Exception:
            # Variations are not available on older compatibility levels.
            pass

    return required


def _disable_mdx_once(dataset, workspace, force_keep: set, opened: list) -> dict:
    """A single pass. Raises if the server rejects the save."""

    changed, skipped = [], []

    with connect_semantic_model(
        dataset=dataset, workspace=workspace, readonly=False
    ) as tom:

        # Kept so the caller can dispose the connection if the save throws.
        opened.append(tom)

        required = _columns_requiring_mdx(tom)

        # all_columns() already excludes system RowNumber columns.
        for col in tom.all_columns():
            key = (col.Parent.Name, col.Name)
            full_name = f"'{key[0]}'[{key[1]}]"

            reason = required.get(key)
            if key in force_keep:
                reason = reason or "rejected by the server on a previous attempt"

            if reason:
                skipped.append((full_name, reason))
                continue

            if not col.IsAvailableInMDX:
                continue

            col.IsAvailableInMDX = False
            changed.append(full_name)

        # Discard the in-memory edits so the automatic SaveChanges() on exit
        # has nothing to commit.
        if WHAT_IF:
            tom.model.UndoLocalChanges()

    return {"changed": changed, "skipped": skipped}


def disable_mdx_for_model(dataset: str, workspace=None) -> dict:
    """Set IsAvailableInMdx = False for all eligible columns in one model."""

    force_keep = set()

    for _ in range(MAX_RETRIES):
        opened = []
        try:
            return _disable_mdx_once(dataset, workspace, force_keep, opened)
        except Exception as e:
            # A failed SaveChanges() skips the library's own Dispose() call,
            # so close the connection here rather than leak one per retry.
            for tom in opened:
                try:
                    tom._tom_server.Dispose()
                except Exception:
                    pass

            # Object names come back wrapped in tags, e.g. '<oii>Date</oii>'.
            message = str(e).replace("<oii>", "").replace("</oii>", "")
            match = _ERROR_COLUMN_RE.search(message)

            if not match or "attribute hierarchy" not in message:
                raise

            key = (match.group(1), match.group(2))
            if key in force_keep:
                raise

            force_keep.add(key)
            print(
                f"        retrying without '{key[0]}'[{key[1]}] "
                f"(the server requires its attribute hierarchy)"
            )

    raise RuntimeError(f"Gave up after {MAX_RETRIES} attempts on '{dataset}'.")


# ---------------------------------------------------------------------------
# Loop over every semantic model in the workspace
# ---------------------------------------------------------------------------
models = fabric.list_datasets(workspace=WORKSPACE)
name_col = "Dataset Name" if "Dataset Name" in models.columns else models.columns[0]

results = []
for name in models[name_col]:

    if name in SKIP_MODELS:
        print(f"[skip] {name}: in SKIP_MODELS")
        continue

    try:
        outcome = disable_mdx_for_model(name, workspace=WORKSPACE)
        print(
            f"[{'what-if' if WHAT_IF else 'ok'}] {name}: "
            f"{len(outcome['changed'])} column(s) set to IsAvailableInMdx=False, "
            f"{len(outcome['skipped'])} skipped"
        )
        for col_name, reason in outcome["skipped"]:
            print(f"        skipped {col_name} ({reason})")

        results.append(
            {
                "Semantic Model": name,
                "Columns Changed": len(outcome["changed"]),
                "Columns Skipped": len(outcome["skipped"]),
                "Status": "What-If" if WHAT_IF else "Updated",
            }
        )
    except Exception as e:
        print(f"[error] {name}: {e}")
        results.append(
            {
                "Semantic Model": name,
                "Columns Changed": 0,
                "Columns Skipped": 0,
                "Status": f"Failed: {e}",
            }
        )

display(pd.DataFrame(results))
